<a href="https://colab.research.google.com/github/michelleasilveira/ST-554-HW8/blob/main/HW8MichelleSilveira.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ST 554**,**HW8**,
Michelle Silveira

*Note: The Homework 8 is a sequence of the Homework 7, and since models will be compared between the HW7 and HW8 i added the HW8 at the end of HW7.*

## Read in and Combine Data

The UCI Machine Learning Repository provides two separate CSV files for wine quality — one for red wines and one for white wines. Both datasets share the same physicochemical features (fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, and alcohol) along with a sensory quality score. The files use semicolons as delimiters, so we read them with `sep=';'`.

After loading both files, we add a `type` column to label each row as either `'red'` or `'white'`, then concatenate them into a single combined dataset. This unified DataFrame will serve as the basis for all subsequent modeling steps.

*Note: the winequality csv files were downloaded from UCI website and uploaded to my content folder in my google drive ( /content)*


In [2]:
import pandas as pd

# File paths
red_path = '/content/winequality-red.csv'
white_path = '/content/winequality-white.csv'

# Read datasets (note the separator ;)
red = pd.read_csv(red_path, sep=';')
white = pd.read_csv(white_path, sep=';')

# Create wine type variable
red['type'] = 'red'
white['type'] = 'white'

# Combine datasets
wine = pd.concat([red, white], axis=0).reset_index(drop=True)
display(wine.head())
display(wine.tail())

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white
6496,6.0,0.21,0.38,0.8,0.020,22.0,98.0,0.98941,3.26,0.32,11.8,6,white


## Split the Data

Before fitting any models, we partition the combined dataset into a **training set (80%)** and a **test set (20%)**. An important consideration here is class imbalance: red wines make up roughly 25% of the data while white wines make up about 75%. To preserve this natural proportion in both splits, we use **stratified sampling** by passing `stratify=wine['type']` to `train_test_split()`. Setting `random_state=42` ensures the split is reproducible. Keeping the test set completely held out until the final evaluation step prevents data leakage and gives us honest performance estimates.

In [3]:
from sklearn.model_selection import train_test_split

# Split data with stratification on wine type
train, test = train_test_split(
    wine,
    test_size=0.2,
    stratify=wine['type'],
    random_state=42
)

# Check proportions
print("Train distribution:")
print(train['type'].value_counts(normalize=True))

print("\nTest distribution:")
print(test['type'].value_counts(normalize=True))

Train distribution:
type
white    0.753896
red      0.246104
Name: proportion, dtype: float64

Test distribution:
type
white    0.753846
red      0.246154
Name: proportion, dtype: float64


## Regression Task — Predicting Alcohol Content

Rather than predicting wine quality, we treat **alcohol content** as our continuous response variable. All remaining numeric features are used as predictors (excluding `type`, which is categorical).

We fit four MLR models of increasing complexity:
- **Basic Linear Regression:** Standard OLS with all numeric predictors, scaled with `StandardScaler`.
- **Polynomial (degree 2):** Adds squared terms for each predictor, capturing nonlinear relationships.
- **Interaction Terms Only:** Includes all pairwise products between predictors without squared terms, modeling how variables jointly influence alcohol.
- **Baseline (no scaling):** Plain `LinearRegression` without preprocessing, serving as a reference.

All models are evaluated with **5-fold cross-validation using RMSE** as the scoring metric.

In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# Define predictors and response
X_train_reg = train.drop(columns=['alcohol', 'type']) # Drop 'type' column as it's non-numeric
y_train_reg = train['alcohol']

# Model 1: Basic Linear
model1 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

# Model 2: Polynomial
model2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

# Model 3: Interaction (via polynomial)
model3 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

# Model 4: Linear regression on a selected subset of predictors
subset_features_reg = ['density', 'residual sugar', 'pH', 'volatile acidity', 'sulphates']
model4 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
# Cross-validation (RMSE)
models = {
    'Linear': model1,
    'Polynomial': model2,
    'Interaction': model3,
    'Subset': model4
}

cv_results = {}

for name, (model, X) in zip(
    ['Linear', 'Polynomial', 'Interaction', 'Subset'],
    [(model1, X_train_reg), (model2, X_train_reg), (model3, X_train_reg), (model4, train[subset_features_reg])]
):
    scores = cross_val_score(model, X, y_train_reg,
                             scoring='neg_root_mean_squared_error', cv=5)
    cv_results[name] = -np.mean(scores)

print("CV RMSE:")
print(cv_results)

CV RMSE:
{'Linear': np.float64(0.5436671099054395), 'Polynomial': np.float64(0.4599142127782347), 'Interaction': np.float64(0.5196600768250005), 'Subset': np.float64(0.7873622639147879)}


### Selecting the Best MLR Model via Cross-Validation

Comparing the four models on 5-fold CV RMSE, the **Polynomial model** achieves the lowest average error (≈0.460), outperforming both the basic linear model (≈0.544) and the interaction-only model (≈0.520). The added squared terms allow the model to capture curvature in the relationship between predictors and alcohol content. We carry this model forward as our best MLR candidate for final test evaluation.

In [5]:
best_model_name = min(cv_results, key=cv_results.get)
best_rmse = cv_results[best_model_name]

print(f"The best model is '{best_model_name}' with an RMSE of {best_rmse:.4f}")

best_model = models[best_model_name]

The best model is 'Polynomial' with an RMSE of 0.4599


### LASSO Regression
• Fit a LASSO model with a set of predictors of your choosing

– Use at least five predictors

– Use CV to select the tuning parameter

We fit a LASSO model using seven predictors. The regularization strength alpha is chosen via 5-fold cross-validation. LASSO can shrink coefficients to exactly zero, acting as a variable selector.

In [6]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

# Choose predictors
predictors_lasso = [
    'fixed acidity',
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'density',
    'pH'
]

X_train_lasso = train[predictors_lasso]
y_train_lasso = train['alcohol']

# LASSO model with CV
lasso_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=42))
])

lasso_model.fit(X_train_lasso, y_train_lasso)

# Best tuning parameter
print("Best alpha (LASSO):", lasso_model.named_steps['lasso'].alpha_)

# Coefficients
coefficients = pd.Series(
    lasso_model.named_steps['lasso'].coef_,
    index=predictors_lasso
)

print("\nCoefficients:")
print(coefficients)

Best alpha (LASSO): 0.0008255721135897018

Coefficients:
fixed acidity       0.829440
volatile acidity    0.266627
citric acid         0.042446
residual sugar      0.840698
chlorides           0.062569
density            -1.780076
pH                  0.540583
dtype: float64


All variables were kept, so all of them help explain alcohol.

Density has the biggest effect and it decreases alcohol.

Residual sugar and fixed acidity increase alcohol the most.

Overall, density is the most important variable.

• Fit a **Ridge Regression model** with a set of predictors of your choosing


– Use at least **five** predictors

– Use **CV** to select the tuning parameter


We fit a Ridge model using the same seven predictors as LASSO. Unlike LASSO, Ridge keeps all coefficients non-zero and shrinks them toward zero. Alpha is chosen via 5-fold cross-validation over a log-spaced grid.

In [7]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# Choose predictors
predictors_ridge = [
    'fixed acidity',
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'density',
    'pH'
]

X_train_ridge = train[predictors_ridge]
y_train_ridge = train['alcohol']

# Candidate alpha values
alphas = np.logspace(-4, 4, 50)

# Ridge model with CV
ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV(alphas=alphas, cv=5))
])

ridge_model.fit(X_train_ridge, y_train_ridge)

# Best tuning parameter
print("Best alpha (Ridge):", ridge_model.named_steps['ridge'].alpha_)

# Coefficients
coefficients = pd.Series(
    ridge_model.named_steps['ridge'].coef_,
    index=predictors_ridge
)

print("\nCoefficients:")
print(coefficients)

Best alpha (Ridge): 3.727593720314938

Coefficients:
fixed acidity       0.829563
volatile acidity    0.267716
citric acid         0.043815
residual sugar      0.841809
chlorides           0.063307
density            -1.780994
pH                  0.541483
dtype: float64


• Fit an Elastic Net model with a set of predictors of your choosing

– Use at least five predictors

– Use CV to select the tuning parameters

In [8]:
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

# Choose predictors
predictors_enet = [
    'fixed acidity',
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'density',
    'pH'
]

X_train_enet = train[predictors_enet]
y_train_enet = train['alcohol']

# Elastic Net model with CV
enet_model = Pipeline([
    ('scaler', StandardScaler()),
    ('enet', ElasticNetCV(cv=5, random_state=42))
])

enet_model.fit(X_train_enet, y_train_enet)

# Best tuning parameters
print("Best alpha (Elastic Net):", enet_model.named_steps['enet'].alpha_)
print("Best l1_ratio:", enet_model.named_steps['enet'].l1_ratio_)

# Coefficients
coefficients = pd.Series(
    enet_model.named_steps['enet'].coef_,
    index=predictors_enet
)

print("\nCoefficients:")
print(coefficients)

Best alpha (Elastic Net): 0.0016511442271794036
Best l1_ratio: 0.5

Coefficients:
fixed acidity       0.822092
volatile acidity    0.265527
citric acid         0.043356
residual sugar      0.831103
chlorides           0.059914
density            -1.768677
pH                  0.536277
dtype: float64


##Test Models

• Using your four selected models, compare their performance on the test set.

– Do so using RMSE as your model metric

– Do so using MAE as your model metric

Note: LASSO, Ridge, and Elastic Net were trained using only the seven selected predictors, while the best MLR (Polynomial) uses all numeric features with degree-2 expansion.


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# Test data
X_test_full = test.drop(columns=['alcohol', 'type'])
y_test = test['alcohol']

# Choose best MLR
best_mlr = model2

# Models to compare
models_final = {
    'Best MLR': (best_mlr, X_train_reg, X_test_full),
    'LASSO': (lasso_model, X_train_lasso, test[predictors_lasso]),
    'Ridge': (ridge_model, X_train_ridge, test[predictors_ridge]),
    'Elastic Net': (enet_model, X_train_enet, test[predictors_enet])
}

results = []

for name, (model, X_train_used, X_test_used) in models_final.items():
    model.fit(X_train_used, y_train_reg)
    preds = model.predict(X_test_used)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    results.append([name, rmse, mae])

results_df = pd.DataFrame(results, columns=['Model', 'RMSE', 'MAE'])
display(results_df)

,Model,RMSE,MAE
0,Best MLR,0.473236,0.342595
1,LASSO,0.591061,0.440918
2,Ridge,0.590992,0.440991
3,Elastic Net,0.591600,0.441480


The Best MLR model performed the best, with the lowest errors.

The other models (LASSO, Ridge, and Elastic Net) all had similar and higher errors.

##Classification Task (Wine Type as Response)

• Repeat the training and testing done previously but use logistic regression models.

• Use log-loss or negative log-loss as your metric for choosing models during the training process

• During the testing portion, compare your models on both log-loss and accuracy

### Four Logistic Regression Models

We fit four logistic regression variants analogous to the MLR models above:
- **Basic:** Standard logistic regression with all available numeric predictors.
- **Polynomial:** Adds squared terms for each predictor.
- **Interaction Only:** Includes all pairwise products between predictors.
- **Subset:** Uses only the six most chemically distinctive predictors (volatile acidity, residual sugar, chlorides, total sulfur dioxide, density, sulphates).

All models are compared using **5-fold CV log-loss** (lower = better calibrated probabilities).

In [10]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
import numpy as np

# Re-create train/test split (stratified on wine type)
train, test = train_test_split(
    wine,
    test_size=0.2,
    stratify=wine['type'],
    random_state=42
)

# Encode wine type: 0 = red, 1 = white
le = LabelEncoder()
y_train_clf = le.fit_transform(train['type'])
y_test_clf  = le.transform(test['type'])

# All numeric predictors (alcohol excluded — it's the regression target)
feature_cols_clf = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
    'density', 'pH', 'sulphates', 'quality'
]

X_train_clf = train[feature_cols_clf]
X_test_clf  = test[feature_cols_clf]

# Focused subset for Model 4
feature_subset_clf = [
    'volatile acidity', 'residual sugar', 'chlorides',
    'total sulfur dioxide', 'density', 'sulphates'
]

# Model 1: Basic logistic regression
log1 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

# Model 2: Polynomial features (degree 2)
log2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=2000, random_state=42))
])

# Model 3: Interaction terms only
log3 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=2000, random_state=42))
])

# Model 4: Subset of most chemically distinctive predictors
log4 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

log_model_configs = {
    'Basic Logistic':       (log1, X_train_clf),
    'Polynomial Logistic':  (log2, X_train_clf),
    'Interaction Logistic': (log3, X_train_clf),
    'Subset Logistic':      (log4, train[feature_subset_clf])
}

cv_log_results = {}
for name, (model, X) in log_model_configs.items():
    scores = cross_val_score(model, X, y_train_clf,
                             scoring='neg_log_loss', cv=5)
    cv_log_results[name] = -np.mean(scores)
    print(f"{name}: CV Log-Loss = {cv_log_results[name]:.4f}")

Basic Logistic: CV Log-Loss = 0.0462
Polynomial Logistic: CV Log-Loss = 0.0364
Interaction Logistic: CV Log-Loss = 0.0409
Subset Logistic: CV Log-Loss = 0.0498


### Selecting the Best Logistic Regression Model

We identify the model with the lowest 5-fold CV log-loss. This is the model whose predicted probabilities most closely match the true class labels on unseen data during training.

In [11]:
best_log_name    = min(cv_log_results, key=cv_log_results.get)
best_log_logloss = cv_log_results[best_log_name]
print(f"Best logistic model: '{best_log_name}' with CV Log-Loss = {best_log_logloss:.4f}")

Best logistic model: 'Polynomial Logistic' with CV Log-Loss = 0.0364


### LASSO Logistic Regression (L1 Penalty)

LASSO logistic regression applies an L1 penalty to the log-likelihood, encouraging sparsity — some coefficients may be driven to exactly zero. We use `LogisticRegressionCV` with `penalty='l1'` and the `liblinear` solver (required for L1), tuning the inverse regularization parameter `C` via 5-fold CV on log-loss. Larger `C` means weaker regularization.

In [12]:
predictors_log_pen = [
    'fixed acidity', 'volatile acidity', 'citric acid',
    'residual sugar', 'chlorides', 'density', 'pH'
]

X_train_log_pen = train[predictors_log_pen]

lasso_log = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 20),
        cv=5,
        penalty='l1',
        solver='liblinear',
        scoring='neg_log_loss',
        random_state=42,
        max_iter=1000
    ))
])

lasso_log.fit(X_train_log_pen, y_train_clf)
print(f"Best C (LASSO Logistic): {lasso_log.named_steps['lasso'].C_[0]:.6f}")

coef_lasso_log = pd.Series(
    lasso_log.named_steps['lasso'].coef_[0],
    index=predictors_log_pen
)
print("\nCoefficients:")
print(coef_lasso_log)

Best C (LASSO Logistic): 1.623777

Coefficients:
fixed acidity      -1.383548
volatile acidity   -1.592436
citric acid         0.237115
residual sugar      3.289635
chlorides          -0.853145
density            -2.395039
pH                 -1.316271
dtype: float64


### Ridge Logistic Regression (L2 Penalty)

Ridge logistic regression applies an L2 penalty, shrinking all coefficients toward zero without eliminating them. This is sklearn's default logistic regression regularization. We use `LogisticRegressionCV` with `penalty='l2'` to find the optimal `C` via 5-fold CV on log-loss, using the same seven predictors as LASSO for direct comparison.

In [13]:
ridge_log = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 20),
        cv=5,
        penalty='l2',
        scoring='neg_log_loss',
        random_state=42,
        max_iter=1000
    ))
])

ridge_log.fit(X_train_log_pen, y_train_clf)
print(f"Best C (Ridge Logistic): {ridge_log.named_steps['ridge'].C_[0]:.6f}")

coef_ridge_log = pd.Series(
    ridge_log.named_steps['ridge'].coef_[0],
    index=predictors_log_pen
)
print("\nCoefficients:")
print(coef_ridge_log)

Best C (Ridge Logistic): 1.623777

Coefficients:
fixed acidity      -1.408023
volatile acidity   -1.580201
citric acid         0.249239
residual sugar      3.198626
chlorides          -0.861202
density            -2.321323
pH                 -1.323899
dtype: float64


### Elastic Net Logistic Regression


Elastic Net combines the L1 and L2 penalties. The l1_ratio parameter controls the balance between LASSO and Ridge behavior. Both alpha and l1_ratio are selected via 5-fold cross-validation.

In [14]:
from sklearn.model_selection import GridSearchCV

enet_log_base = Pipeline([
    ('scaler', StandardScaler()),
    ('enet', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        max_iter=3000,
        random_state=42
    ))
])

param_grid = {
    'enet__C':        np.logspace(-3, 3, 10),
    'enet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_log = GridSearchCV(
    enet_log_base,
    param_grid,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1
)

enet_log.fit(X_train_log_pen, y_train_clf)
print("Best params (Elastic Net Logistic):", enet_log.best_params_)
print(f"Best CV Log-Loss: {-enet_log.best_score_:.4f}")

Best params (Elastic Net Logistic): {'enet__C': np.float64(2.154434690031882), 'enet__l1_ratio': 0.1}
Best CV Log-Loss: 0.0953


## Testing Classification Models

Classification Task - Wine Type as Response

We repeat the same structure as the regression section but now predict wine type (red or white) as a binary outcome. Log-loss is used to select models during training. On the test set we compare models on both log-loss and accuracy.

- Log-Loss Measures how well-calibrated the predicted probabilities are — lower is better.
- Accuracy Proportion of wines correctly classified as red or white.

Wine type is chemically quite distinctive (e.g., white wines have much higher residual sugar and sulfur dioxide), so we expect high accuracy across all models.

In [16]:
from sklearn.metrics import log_loss, accuracy_score

# Refit all 4 logistic models on their respective training data
log_test_configs = {
    'Basic Logistic':       (log1, X_train_clf,            X_test_clf),
    'Polynomial Logistic':  (log2, X_train_clf,            X_test_clf),
    'Interaction Logistic': (log3, X_train_clf,            X_test_clf),
    'Subset Logistic':      (log4, train[feature_subset_clf], test[feature_subset_clf])
}

for name, (model, X_tr, _) in log_test_configs.items():
    model.fit(X_tr, y_train_clf)

clf_results = []

# Best MLR-style logistic model
best_log_model, best_X_tr, best_X_te = log_test_configs[best_log_name]
proba = best_log_model.predict_proba(best_X_te)
pred  = best_log_model.predict(best_X_te)
clf_results.append([f'Best LogReg ({best_log_name})',
                    log_loss(y_test_clf, proba),
                    accuracy_score(y_test_clf, pred)])

# LASSO logistic
proba = lasso_log.predict_proba(test[predictors_log_pen])
pred  = lasso_log.predict(test[predictors_log_pen])
clf_results.append(['LASSO Logistic',
                    log_loss(y_test_clf, proba),
                    accuracy_score(y_test_clf, pred)])

# Ridge logistic
proba = ridge_log.predict_proba(test[predictors_log_pen])
pred  = ridge_log.predict(test[predictors_log_pen])
clf_results.append(['Ridge Logistic',
                    log_loss(y_test_clf, proba),
                    accuracy_score(y_test_clf, pred)])

# Elastic Net logistic
proba = enet_log.predict_proba(test[predictors_log_pen])
pred  = enet_log.predict(test[predictors_log_pen])
clf_results.append(['Elastic Net Logistic',
                    log_loss(y_test_clf, proba),
                    accuracy_score(y_test_clf, pred)])

clf_results_df = pd.DataFrame(clf_results, columns=['Model', 'Log-Loss', 'Accuracy'])
display(clf_results_df)

,Model,Log-Loss,Accuracy
0,Best LogReg (Polynomial Logistic),0.020520,0.994615
1,LASSO Logistic,0.078635,0.975385
2,Ridge Logistic,0.078714,0.975385
3,Elastic Net Logistic,0.078606,0.975385


All logistic regression models achieve very high accuracy, confirming that wine type is highly separable from physicochemical measurements. The penalized models (LASSO, Ridge, Elastic Net) perform competitively despite using only seven predictors — key variables like residual sugar total sulfur dioxide and volatile acidity carry strong discriminative signal. The model selected via CV log-loss generalizes well to the test set, validating our training-phase model selection strategy. Log-loss values near zero indicate that the models are producing well-calibrated probabilities, not just correct classifications.

# Homework 8
## Regression Task - Tree Models

We now add two tree-based models to compare against the HW7 regression models. A regression tree is fit with CV to choose max_depth and min_samples_leaf. A random forest is fit with CV to choose max_features.

In [18]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

param_grid_tree = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 5, 10, 20, 50]
}

tree_reg = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_tree,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

tree_reg.fit(X_train_reg, y_train_reg)
print("Best params (Regression Tree):", tree_reg.best_params_)
print(f"Best CV RMSE: {-tree_reg.best_score_:.4f}")

Best params (Regression Tree): {'max_depth': None, 'min_samples_leaf': 5}
Best CV RMSE: 0.5658


The best max_depth and min_samples_leaf are selected by 5-fold CV. Deeper trees tend to overfit while very shallow trees underfit, so CV finds the right balance.

In [19]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rf = {
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7]
}

rf_reg = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=42),
    param_grid_rf,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

rf_reg.fit(X_train_reg, y_train_reg)
print("Best params (Random Forest):", rf_reg.best_params_)
print(f"Best CV RMSE: {-rf_reg.best_score_:.4f}")

Best params (Random Forest): {'max_features': 0.7}
Best CV RMSE: 0.4241


Random forests reduce variance compared to a single tree by averaging many trees trained on bootstrapped samples. max_features controls how many predictors are considered at each split, adding further randomness.

## Test Models - Regression (all HW7 and HW8 models)

We compare all regression models on the test set using RMSE and MAE, including the four models from HW7.

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

y_test = test['alcohol']
X_test_full = test.drop(columns=['alcohol', 'type'])

# Refit HW7 models on training data
model2.fit(X_train_reg, y_train_reg)
lasso_model.fit(train[predictors_lasso], y_train_reg)
ridge_model.fit(train[predictors_ridge], y_train_reg)
enet_model.fit(train[predictors_enet], y_train_reg)

all_reg_models = {
    'Best MLR (Polynomial)': (model2, X_test_full),
    'LASSO':                  (lasso_model, test[predictors_lasso]),
    'Ridge':                  (ridge_model, test[predictors_ridge]),
    'Elastic Net':            (enet_model, test[predictors_enet]),
    'Regression Tree':        (tree_reg, X_test_full),
    'Random Forest':          (rf_reg, X_test_full)
}

results_hw8 = []
for name, (model, X_te) in all_reg_models.items():
    preds = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results_hw8.append([name, rmse, mae])

results_hw8_df = pd.DataFrame(results_hw8, columns=['Model', 'RMSE', 'MAE'])
display(results_hw8_df)

,Model,RMSE,MAE
0,Best MLR (Polynomial),0.473236,0.342595
1,LASSO,0.591061,0.440918
2,Ridge,0.590992,0.440991
3,Elastic Net,0.591600,0.441480
4,Regression Tree,0.546751,0.376844
5,Random Forest,0.404802,0.269587


The random forest outperformed all other models with the lowest RMSE (0.405) and MAE (0.270). The regression tree fell between the penalized models and the polynomial MLR. The polynomial MLR remains the second-best single model due to its nonlinear feature expansion.

## Classification Task - Tree Models

We repeat the classification task using a decision tree and a random forest. Log-loss is used for model selection during CV. The final test compares all models from HW7 and HW8 on log-loss and accuracy.

In [21]:
from sklearn.tree import DecisionTreeClassifier

param_grid_tree_clf = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 5, 10, 20, 50]
}

tree_clf = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_tree_clf,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1
)

tree_clf.fit(X_train_clf, y_train_clf)
print("Best params (Classification Tree):", tree_clf.best_params_)
print(f"Best CV Log-Loss: {-tree_clf.best_score_:.4f}")

Best params (Classification Tree): {'max_depth': 3, 'min_samples_leaf': 50}
Best CV Log-Loss: 0.1180


In [22]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf_clf = {
    'max_features': ['sqrt', 'log2', 0.3, 0.5]
}

rf_clf = GridSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    param_grid_rf_clf,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1
)

rf_clf.fit(X_train_clf, y_train_clf)
print("Best params (RF Classifier):", rf_clf.best_params_)
print(f"Best CV Log-Loss: {-rf_clf.best_score_:.4f}")

Best params (RF Classifier): {'max_features': 'sqrt'}
Best CV Log-Loss: 0.0444


Test Models - Classification (all HW7 and HW8 models)

We evaluate all classification models on the test set. Log-loss and accuracy are reported for every model including the logistic regression models from HW7.

In [23]:
from sklearn.metrics import log_loss, accuracy_score

# Refit HW7 logistic models
log2.fit(X_train_clf, y_train_clf)
lasso_log.fit(train[predictors_log_pen], y_train_clf)
ridge_log.fit(train[predictors_log_pen], y_train_clf)
enet_log.fit(train[predictors_log_pen], y_train_clf)

all_clf_models = [
    ('Polynomial Logistic',   log2,     X_test_clf),
    ('LASSO Logistic',        lasso_log, test[predictors_log_pen]),
    ('Ridge Logistic',        ridge_log, test[predictors_log_pen]),
    ('Elastic Net Logistic',  enet_log,  test[predictors_log_pen]),
    ('Classification Tree',   tree_clf,  X_test_clf),
    ('Random Forest',         rf_clf,    X_test_clf)
]

clf_results_hw8 = []
for name, model, X_te in all_clf_models:
    proba = model.predict_proba(X_te)
    pred  = model.predict(X_te)
    clf_results_hw8.append([name,
                             log_loss(y_test_clf, proba),
                             accuracy_score(y_test_clf, pred)])

clf_results_hw8_df = pd.DataFrame(clf_results_hw8, columns=['Model', 'Log-Loss', 'Accuracy'])
display(clf_results_hw8_df)

,Model,Log-Loss,Accuracy
0,Polynomial Logistic,0.020520,0.994615
1,LASSO Logistic,0.078635,0.975385
2,Ridge Logistic,0.078714,0.975385
3,Elastic Net Logistic,0.078606,0.975385
4,Classification Tree,0.084887,0.972308
5,Random Forest,0.024813,0.996154


The random forest achieved the highest accuracy (99.6%) and a very low log-loss (0.025), closely followed by the polynomial logistic regression (99.5%, log-loss 0.021). The classification tree had the highest log-loss (0.085) and lowest accuracy (97.2%), likely because very shallow trees (max_depth=3 was selected) produce rough probability estimates. Overall, ensemble methods and logistic regression with polynomial features are the top performers for this classification task.